# 👥 IA Neurosimbólica — Gestión de Talento en Recursos Humanos

**Caso de uso:** El departamento de RH de una empresa mediana (~500 empleados) necesita un sistema que ayude en dos decisiones críticas:
1. **Riesgo de rotación:** identificar empleados en riesgo de salir antes de que renuncien.
2. **Elegibilidad para promoción:** recomendar candidatos a ascenso de forma justa y auditada.

El sistema debe ser justo, explicable y alineado con la política interna de RH — no puede simplemente arrojar un número y ya.

---
### ¿Por qué IA Neurosimbólica en RH?

| Enfoque | Problema |
|---------|----------|
| Solo ML | Puede ser sesgado; no explica por qué recomienda a alguien |
| Solo reglas | Rígido; no detecta patrones de comportamiento sutil |
| **Neurosimbólico** | Detecta señales tempranas (ML) + aplica política justa y explicable (reglas) |

> **Datos:** Sintéticos pero realistas. Variables basadas en estudios de retención de talento (Gallup, LinkedIn, SHRM).

## 📦 Celda 1 — Librerías

Importamos las mismas herramientas que en el notebook de manufactura. En este caso usamos además `LabelEncoder` para convertir variables categóricas (departamento, nivel, etc.) a formato numérico que el modelo de ML pueda procesar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(2024)
print('✅ Librerías cargadas correctamente')

## 🏗️ Celda 2 — Generación del dataset de empleados

Simulamos el perfil de 500 empleados con variables que, según la literatura de RH, son predictivas de rotación y de potencial de crecimiento:

| Variable | Tipo | Descripción |
|----------|------|-------------|
| `antiguedad_anios` | Numérico | Años en la empresa |
| `evaluacion_desempeno` | 1–5 | Último resultado de evaluación |
| `satisfaccion_laboral` | 1–10 | Encuesta de clima |
| `horas_extra_mes` | Numérico | Promedio mensual de horas extra |
| `capacitaciones_ultimo_anio` | Entero | Cursos completados |
| `incidencias_disciplinarias` | Entero | Faltas o amonestaciones en 12 meses |
| `meses_ultimo_aumento` | Entero | Meses desde el último incremento salarial |
| `departamento` | Categórico | Área de trabajo |
| `nivel_puesto` | Categórico | Junior / Semi / Senior / Lead |
| `rotacion` | 0/1 | **Variable objetivo**: si el empleado renunció en los siguientes 6 meses |

In [ ]:
N = 500

departamentos = ['Operaciones', 'Ventas', 'Tecnología', 'Finanzas', 'RH', 'Logística']
niveles       = ['Junior', 'Semi-Senior', 'Senior', 'Lead']

antiguedad         = np.random.exponential(4, N).clip(0.5, 20)   # años
evaluacion         = np.random.choice([1,2,3,4,5], N, p=[0.05,0.15,0.35,0.30,0.15])
satisfaccion       = np.random.normal(6.5, 1.8, N).clip(1, 10)
horas_extra        = np.random.poisson(12, N).clip(0, 60)         # horas/mes
capacitaciones     = np.random.poisson(2, N).clip(0, 8)
incidencias        = np.random.poisson(0.4, N).clip(0, 5)
meses_ultimo_aum   = np.random.randint(1, 36, N)
depto              = np.random.choice(departamentos, N)
nivel              = np.random.choice(niveles, N, p=[0.30, 0.35, 0.25, 0.10])

# Rotación: mayor riesgo si baja satisfacción, muchas horas extra,
# mucho tiempo sin aumento y baja evaluación
prob_rotacion = (
    0.05
    + 0.12 * (satisfaccion < 5).astype(float)
    + 0.10 * (horas_extra > 25).astype(float)
    + 0.08 * (meses_ultimo_aum > 24).astype(float)
    + 0.10 * (evaluacion <= 2).astype(float)
    + 0.07 * (incidencias >= 2).astype(float)
    - 0.05 * (evaluacion >= 4).astype(float)
    - 0.04 * (capacitaciones >= 3).astype(float)
).clip(0, 0.85)

rotacion = (np.random.uniform(0, 1, N) < prob_rotacion).astype(int)

df = pd.DataFrame({
    'empleado_id':            [f'EMP-{i:04d}' for i in range(N)],
    'departamento':           depto,
    'nivel_puesto':           nivel,
    'antiguedad_anios':       np.round(antiguedad, 1),
    'evaluacion_desempeno':   evaluacion,
    'satisfaccion_laboral':   np.round(satisfaccion, 1),
    'horas_extra_mes':        horas_extra,
    'capacitaciones_anio':    capacitaciones,
    'incidencias_disc':       incidencias,
    'meses_ultimo_aumento':   meses_ultimo_aum,
    'rotacion':               rotacion
})

print(f'Dataset generado: {N} empleados')
print(f'  → Rotaron en 6 meses: {rotacion.sum()} ({rotacion.mean()*100:.1f}%)')
print(f'  → Permanecieron:      {(1-rotacion).sum()} ({(1-rotacion).mean()*100:.1f}%)')
df.head(6)

## 📊 Celda 3 — Análisis exploratorio: ¿qué factores se asocian a la rotación?

Antes de entrenar el modelo, visualizamos las diferencias entre empleados que rotaron y los que se quedaron. Esto nos ayuda a validar que los datos tienen sentido y a comunicarle a la dirección de RH qué variables son más relevantes.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Exploración: factores asociados a rotación de personal\nAzul = se quedó | Rojo = rotó',
             fontsize=12)

vars_num = ['satisfaccion_laboral', 'horas_extra_mes', 'meses_ultimo_aumento',
            'evaluacion_desempeno', 'capacitaciones_anio', 'antiguedad_anios']

for ax, var in zip(axes.flatten(), vars_num):
    for clase, color, label in [(0,'#3B82F6','Permanece'), (1,'#EF4444','Rotó')]:
        ax.hist(df[df['rotacion']==clase][var], bins=20, alpha=0.55,
                color=color, label=label, density=True)
    ax.set_title(var.replace('_', ' '), fontsize=9)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Tasa de rotación por departamento
fig, ax = plt.subplots(figsize=(8, 4))
tasa_depto = df.groupby('departamento')['rotacion'].mean().sort_values(ascending=True)
bars = ax.barh(tasa_depto.index, tasa_depto.values * 100, color='#6366F1', edgecolor='white')
ax.set_xlabel('Tasa de rotación (%)')
ax.set_title('Tasa de rotación por departamento', fontsize=11)
for bar, val in zip(bars, tasa_depto.values):
    ax.text(val*100 + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val*100:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 🧠 Celda 4 — CAPA NEURAL: Modelo de predicción de riesgo de rotación

Usamos un **Gradient Boosting Classifier** (más potente que Random Forest para datos tabulares desbalanceados). El modelo aprende qué combinaciones de variables predicen la salida de un empleado.

Antes del entrenamiento codificamos las variables categóricas (`departamento`, `nivel_puesto`) a números, porque el modelo solo trabaja con valores numéricos.

In [ ]:
# Codificación de variables categóricas
df_ml = df.copy()
le_depto = LabelEncoder()
le_nivel = LabelEncoder()
df_ml['departamento_enc'] = le_depto.fit_transform(df_ml['departamento'])
df_ml['nivel_enc']         = le_nivel.fit_transform(df_ml['nivel_puesto'])

features = ['antiguedad_anios', 'evaluacion_desempeno', 'satisfaccion_laboral',
            'horas_extra_mes', 'capacitaciones_anio', 'incidencias_disc',
            'meses_ultimo_aumento', 'departamento_enc', 'nivel_enc']

X = df_ml[features]
y = df_ml['rotacion']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# Entrenamiento — class_weight para compensar el desbalance de clases
modelo_gb = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.08,
    max_depth=4, subsample=0.8, random_state=42
)
modelo_gb.fit(X_train_sc, y_train)

y_pred = modelo_gb.predict(X_test_sc)
y_prob = modelo_gb.predict_proba(X_test_sc)[:, 1]

print('=== CAPA NEURAL: Desempeño del modelo ===')
print(classification_report(y_test, y_pred, target_names=['Permanece','Rota']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

# Importancia de variables
importancias = pd.Series(modelo_gb.feature_importances_, index=features).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 4))
importancias.plot(kind='barh', ax=ax, color='#6366F1', edgecolor='white')
ax.set_title('Importancia de variables — Capa Neural (Gradient Boosting)', fontsize=11)
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

## 🔗 Celda 5 — CAPA DE INTEGRACIÓN: De probabilidades a perfil simbólico del empleado

La capa de integración convierte el output del modelo en un perfil comprensible. Para cada empleado construimos:
- **Nivel de riesgo de rotación** (basado en la probabilidad)
- **Señales de alerta específicas** (cada factor que contribuye al riesgo)
- **Perfil de potencial de promoción** (basado en métricas objetivas)

Estos hechos simbólicos son los que el motor de reglas usará para proponer acciones concretas.

In [ ]:
def extraer_perfil_empleado(fila, prob_rotacion):
    """
    Capa de integración: construye el perfil simbólico del empleado
    a partir de las métricas + probabilidad del modelo neuronal.
    """
    perfil = {}

    # --- Riesgo de rotación (del modelo neuronal) ---
    if   prob_rotacion < 0.20: perfil['riesgo_rotacion'] = 'bajo'
    elif prob_rotacion < 0.45: perfil['riesgo_rotacion'] = 'moderado'
    elif prob_rotacion < 0.65: perfil['riesgo_rotacion'] = 'alto'
    else:                       perfil['riesgo_rotacion'] = 'critico'

    # --- Señales de alerta (hechos simbólicos individuales) ---
    perfil['satisfaccion_baja']        = fila['satisfaccion_laboral'] < 5.0
    perfil['sobrecarga_trabajo']       = fila['horas_extra_mes'] > 25
    perfil['sin_aumento_largo_tiempo'] = fila['meses_ultimo_aumento'] > 18
    perfil['evaluacion_critica']       = fila['evaluacion_desempeno'] <= 2
    perfil['incidencias_recurrentes']  = fila['incidencias_disc'] >= 2
    perfil['bajo_desarrollo']          = fila['capacitaciones_anio'] == 0

    # --- Conteo de alertas activas ---
    alertas = ['satisfaccion_baja', 'sobrecarga_trabajo', 'sin_aumento_largo_tiempo',
               'evaluacion_critica', 'incidencias_recurrentes', 'bajo_desarrollo']
    perfil['n_alertas'] = sum(perfil[a] for a in alertas)

    # --- Perfil de potencial de promoción ---
    perfil['alto_desempeno']      = fila['evaluacion_desempeno'] >= 4
    perfil['experiencia_suficiente'] = fila['antiguedad_anios'] >= 2.0
    perfil['activo_en_formacion'] = fila['capacitaciones_anio'] >= 2
    perfil['sin_incidencias']     = fila['incidencias_disc'] == 0
    perfil['nivel_actual']        = fila['nivel_puesto']

    return perfil

# Demostración con empleado de ejemplo
idx_demo  = 0
fila_demo = X_test.iloc[idx_demo]
fila_orig = df_ml.iloc[X_test.index[idx_demo]]
prob_demo = y_prob[idx_demo]

perfil_demo = extraer_perfil_empleado(fila_orig, prob_demo)

print(f'=== CAPA DE INTEGRACIÓN: Perfil del empleado {df.iloc[X_test.index[idx_demo]]["empleado_id"]} ===')
print(f'Probabilidad de rotación (modelo): {prob_demo:.3f}')
print()
for hecho, valor in perfil_demo.items():
    if isinstance(valor, bool):
        icono = '⚠️ ' if valor else '✅'
        print(f'  {icono}  {hecho}: {valor}')
    else:
        print(f'  📊  {hecho}: {valor}')

## 📋 Celda 6 — CAPA SIMBÓLICA: Motor de decisiones de RH

El motor de reglas de RH codifica la **política de retención y promoción de la empresa**. Genera dos salidas:

**A) Acción de retención** (si hay riesgo de rotación):
- `ALERTA_INMEDIATA`: intervención urgente del manager
- `REVISAR_COMPENSACION`: iniciar proceso de revisión salarial
- `PLAN_BIENESTAR`: intervención de clima y balance trabajo-vida
- `MONITOREAR`: seguimiento periódico sin acción inmediata

**B) Recomendación de promoción** (evaluación de elegibilidad):
- `CANDIDATO_PROMOTION`: cumple todos los criterios
- `EN_DESARROLLO`: cerca de cumplirlos, necesita X
- `NO_ELEGIBLE_AUN`: falta tiempo o métricas
- `REVISION_DISCIPLINARIA`: no elegible por incidencias

In [ ]:
def motor_reglas_rh(perfil, fila):
    """
    Motor de razonamiento simbólico de RH.
    Devuelve: (accion_retencion, justif_retencion, accion_promocion, justif_promocion)
    """
    sat  = fila['satisfaccion_laboral']
    hrs  = fila['horas_extra_mes']
    eval_ = fila['evaluacion_desempeno']
    ant  = fila['antiguedad_anios']
    ult  = fila['meses_ultimo_aumento']
    caps = fila['capacitaciones_anio']
    inc  = fila['incidencias_disc']
    nivel = fila['nivel_puesto']

    # ══════════════════════════════════════════════════════
    # BLOQUE A: Acción de retención
    # ══════════════════════════════════════════════════════

    # Regla A1: Riesgo crítico con múltiples señales — intervención urgente
    if perfil['riesgo_rotacion'] == 'critico' and perfil['n_alertas'] >= 3:
        accion_ret   = 'ALERTA_INMEDIATA'
        justif_ret   = (
            f'Riesgo CRÍTICO de salida. Satisfacción: {sat:.1f}/10, '
            f'horas extra: {hrs}h/mes, {perfil["n_alertas"]} señales activas. '
            f'Se requiere reunión urgente manager + RH en las próximas 48 horas.'
        )

    # Regla A2: Sin aumento prolongado + riesgo alto
    elif perfil['sin_aumento_largo_tiempo'] and perfil['riesgo_rotacion'] in ('alto','critico'):
        accion_ret   = 'REVISAR_COMPENSACION'
        justif_ret   = (
            f'{ult} meses sin incremento salarial. El mercado laboral indica '
            f'riesgo de oferta externa. Iniciar proceso de revisión de banda '
            f'salarial con Compensaciones.'
        )

    # Regla A3: Sobrecarga de trabajo + satisfacción baja
    elif perfil['sobrecarga_trabajo'] and perfil['satisfaccion_baja']:
        accion_ret   = 'PLAN_BIENESTAR'
        justif_ret   = (
            f'{hrs}h de horas extra/mes con satisfacción de {sat:.1f}/10. '
            f'Patrón de agotamiento (burnout). Activar programa de bienestar '
            f'y revisar carga de trabajo con el área.'
        )

    # Regla A4: Riesgo moderado — monitoreo
    elif perfil['riesgo_rotacion'] == 'moderado':
        accion_ret   = 'MONITOREAR'
        justif_ret   = (
            f'Riesgo moderado ({perfil["n_alertas"]} alerta(s)). '
            f'Incluir en próxima encuesta de pulso y dar seguimiento trimestral.'
        )

    # Regla A5: Riesgo bajo — sin acción
    else:
        accion_ret   = 'SIN_ACCION'
        justif_ret   = (
            f'Perfil estable. Riesgo de rotación BAJO. '
            f'Continuar con seguimiento regular de desempeño.'
        )

    # ══════════════════════════════════════════════════════
    # BLOQUE B: Elegibilidad para promoción
    # ══════════════════════════════════════════════════════

    # Regla B1: Incidencias disciplinarias — bloqueante
    if perfil['incidencias_recurrentes']:
        accion_prom  = 'REVISION_DISCIPLINARIA'
        justif_prom  = (
            f'{inc} incidencia(s) disciplinaria(s) en los últimos 12 meses. '
            f'La política PL-RH-012 requiere 12 meses sin incidencias '
            f'para ser elegible a promoción.'
        )

    # Regla B2: Cumple todos los criterios de promoción
    elif (perfil['alto_desempeno'] and perfil['experiencia_suficiente']
          and perfil['activo_en_formacion'] and perfil['sin_incidencias']
          and nivel != 'Lead'):  # ya está en el nivel máximo si es Lead
        accion_prom  = 'CANDIDATO_PROMOCION'
        justif_prom  = (
            f'Evaluación {eval_}/5, {ant:.1f} años de antigüedad, '
            f'{caps} capacitación(es) reciente(s), sin incidencias. '
            f'Cumple todos los requisitos de PL-RH-007 para ascenso de '
            f'{nivel} al siguiente nivel.'
        )

    # Regla B3: Nivel máximo alcanzado
    elif nivel == 'Lead':
        accion_prom  = 'NIVEL_MAXIMO'
        justif_prom  = (
            f'El empleado ya ocupa el nivel Lead (máximo en banda individual). '
            f'Canalizar hacia programa de desarrollo directivo si aplica.'
        )

    # Regla B4: Buen desempeño pero necesita más formación
    elif perfil['alto_desempeno'] and not perfil['activo_en_formacion']:
        accion_prom  = 'EN_DESARROLLO'
        justif_prom  = (
            f'Evaluación {eval_}/5 excelente, pero solo {caps} capacitación(es) '
            f'en el año. La política PL-RH-007 requiere mínimo 2. '
            f'Inscribir en plan de formación para próxima ventana de promoción.'
        )

    # Regla B5: No elegible aún
    else:
        accion_prom  = 'NO_ELEGIBLE_AUN'
        justif_prom  = (
            f'Evaluación {eval_}/5 y {ant:.1f} años de antigüedad. '
            f'Continuar con plan de desarrollo individual. '
            f'Re-evaluar en la próxima revisión semestral.'
        )

    return accion_ret, justif_ret, accion_prom, justif_prom

# Prueba con el empleado de ejemplo
accion_r, just_r, accion_p, just_p = motor_reglas_rh(perfil_demo, fila_orig)

iconos_ret  = {'ALERTA_INMEDIATA':'🚨','REVISAR_COMPENSACION':'💰','PLAN_BIENESTAR':'🧘',
               'MONITOREAR':'👁️','SIN_ACCION':'✅'}
iconos_prom = {'CANDIDATO_PROMOCION':'🌟','EN_DESARROLLO':'📈','NO_ELEGIBLE_AUN':'⏳',
               'REVISION_DISCIPLINARIA':'⛔','NIVEL_MAXIMO':'🏆'}

print(f'=== CAPA SIMBÓLICA: Decisiones de RH ===')
print(f'{iconos_ret.get(accion_r,"📋")} RETENCIÓN : {accion_r}')
print(f'   {just_r}')
print()
print(f'{iconos_prom.get(accion_p,"📋")} PROMOCIÓN  : {accion_p}')
print(f'   {just_p}')

## 🔄 Celda 7 — Pipeline completo: procesar todos los empleados

Ejecutamos el pipeline completo para los 125 empleados del conjunto de prueba. Este sería el proceso que RH correría mensualmente para generar su reporte de talento.

In [ ]:
resultados = []

for i, (idx, fila_x) in enumerate(X_test.iterrows()):
    fila_completa = df_ml.iloc[idx]
    prob = y_prob[i]

    perfil = extraer_perfil_empleado(fila_completa, prob)
    accion_r, just_r, accion_p, just_p = motor_reglas_rh(perfil, fila_completa)

    resultados.append({
        'empleado_id':         fila_completa['empleado_id'],
        'departamento':        fila_completa['departamento'],
        'nivel_puesto':        fila_completa['nivel_puesto'],
        'prob_rotacion':       round(prob, 3),
        'riesgo_rotacion':     perfil['riesgo_rotacion'],
        'n_alertas':           perfil['n_alertas'],
        'accion_retencion':    accion_r,
        'accion_promocion':    accion_p,
        'justif_retencion':    just_r,
        'justif_promocion':    just_p,
        'rotacion_real':       y_test.iloc[i]
    })

df_rh = pd.DataFrame(resultados)

print('=== RESUMEN MENSUAL RH ===')
print('\n— Acciones de Retención —')
print(df_rh['accion_retencion'].value_counts().to_string())
print('\n— Acciones de Promoción —')
print(df_rh['accion_promocion'].value_counts().to_string())
print(f'\nTotal empleados evaluados: {len(df_rh)}')

## 📊 Celda 8 — Dashboard ejecutivo de RH

El dashboard que vería el Director de RH: distribución de acciones, mapa de riesgo por departamento y lista de candidatos a promoción. Toda la información está **justificada por las reglas**, no solo por el score del modelo.

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Dashboard Ejecutivo — IA Neurosimbólica en RH', fontsize=14, fontweight='bold')

# ── Gráfico 1: Acciones de retención ──────────────────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
conteo_ret = df_rh['accion_retencion'].value_counts()
colores_ret = {'ALERTA_INMEDIATA':'#DC2626','REVISAR_COMPENSACION':'#D97706',
               'PLAN_BIENESTAR':'#7C3AED','MONITOREAR':'#2563EB','SIN_ACCION':'#16A34A'}
bars = ax1.bar(range(len(conteo_ret)), conteo_ret.values,
               color=[colores_ret.get(k,'gray') for k in conteo_ret.index],
               edgecolor='white')
ax1.set_xticks(range(len(conteo_ret)))
ax1.set_xticklabels([k.replace('_','\n') for k in conteo_ret.index], fontsize=8)
ax1.set_title('Acciones de retención', fontsize=10)
for bar, val in zip(bars, conteo_ret.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             str(val), ha='center', fontsize=9, fontweight='bold')

# ── Gráfico 2: Acciones de promoción ─────────────────────────────────────────
ax2 = fig.add_subplot(2, 3, 2)
conteo_prom = df_rh['accion_promocion'].value_counts()
colores_prom = {'CANDIDATO_PROMOCION':'#16A34A','EN_DESARROLLO':'#2563EB',
                'NO_ELEGIBLE_AUN':'#6B7280','REVISION_DISCIPLINARIA':'#DC2626','NIVEL_MAXIMO':'#D97706'}
bars2 = ax2.bar(range(len(conteo_prom)), conteo_prom.values,
                color=[colores_prom.get(k,'gray') for k in conteo_prom.index],
                edgecolor='white')
ax2.set_xticks(range(len(conteo_prom)))
ax2.set_xticklabels([k.replace('_','\n') for k in conteo_prom.index], fontsize=7)
ax2.set_title('Recomendaciones de promoción', fontsize=10)
for bar, val in zip(bars2, conteo_prom.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             str(val), ha='center', fontsize=9, fontweight='bold')

# ── Gráfico 3: Riesgo por departamento ───────────────────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
riesgo_depto = df_rh.groupby('departamento')['prob_rotacion'].mean().sort_values(ascending=True)
colores_riesgo = ['#22C55E' if v < 0.25 else '#F59E0B' if v < 0.45 else '#EF4444'
                  for v in riesgo_depto.values]
ax3.barh(riesgo_depto.index, riesgo_depto.values, color=colores_riesgo, edgecolor='white')
ax3.set_xlabel('Riesgo promedio de rotación')
ax3.set_title('Riesgo de rotación\npor departamento', fontsize=10)
for i, (idx, val) in enumerate(riesgo_depto.items()):
    ax3.text(val + 0.005, i, f'{val:.2f}', va='center', fontsize=8)

# ── Gráfico 4: Heatmap riesgo × alertas ──────────────────────────────────────
ax4 = fig.add_subplot(2, 3, 4)
orden_riesgo = ['bajo','moderado','alto','critico']
tabla = pd.crosstab(df_rh['riesgo_rotacion'], df_rh['n_alertas']).reindex(
    index=orden_riesgo, fill_value=0)
sns.heatmap(tabla, ax=ax4, annot=True, fmt='d', cmap='YlOrRd', cbar=False)
ax4.set_title('Riesgo neural × N° alertas simbólicas', fontsize=10)
ax4.set_xlabel('Número de alertas activas')
ax4.set_ylabel('Riesgo neuronal')

# ── Gráfico 5: Distribución de probabilidades de rotación ────────────────────
ax5 = fig.add_subplot(2, 3, 5)
niveles_col = {'bajo':'#22C55E','moderado':'#F59E0B','alto':'#F97316','critico':'#EF4444'}
for niv, color in niveles_col.items():
    sub = df_rh[df_rh['riesgo_rotacion']==niv]['prob_rotacion']
    if len(sub) > 0:
        ax5.hist(sub, bins=10, alpha=0.6, color=color,
                 label=f'{niv.capitalize()} (n={len(sub)})', density=True)
ax5.set_title('Distribución de riesgo\npor nivel neuronal', fontsize=10)
ax5.set_xlabel('Probabilidad de rotación')
ax5.legend(fontsize=8)

# ── Texto: Top candidatos a promoción ────────────────────────────────────────
ax6 = fig.add_subplot(2, 3, 6)
ax6.axis('off')
candidatos = df_rh[df_rh['accion_promocion']=='CANDIDATO_PROMOCION'][['empleado_id','departamento','nivel_puesto','prob_rotacion']].head(6)
if len(candidatos) > 0:
    ax6.set_title('Top candidatos a promoción', fontsize=10, pad=10)
    tabla_vals = [candidatos.columns.tolist()] + candidatos.values.tolist()
    t = ax6.table(cellText=candidatos.values, colLabels=candidatos.columns,
                  cellLoc='center', loc='center', bbox=[0,0,1,1])
    t.auto_set_font_size(False)
    t.set_fontsize(8)
    for (row, col), cell in t.get_celld().items():
        if row == 0:
            cell.set_facecolor('#6366F1')
            cell.set_text_props(color='white', weight='bold')
        elif row % 2 == 0:
            cell.set_facecolor('#F3F4F6')

plt.tight_layout()
plt.show()

## 🔍 Celda 9 — Ficha de empleado individual: trazabilidad para auditoría

Si en algún momento un empleado o el sindicato cuestiona una decisión de RH, el sistema puede generar la ficha completa de trazabilidad: **qué midió, qué detectó el modelo, qué regla aplicó y qué política lo fundamenta**.

Esto es fundamental en un contexto de cumplimiento y equidad — RH puede demostrar que la decisión no fue arbitraria.

In [ ]:
def ficha_trazabilidad_rh(empleado_id, df_rh, df_ml):
    """Genera la ficha completa de trazabilidad para auditoría de RH."""

    res = df_rh[df_rh['empleado_id'] == empleado_id]
    if len(res) == 0:
        print(f'Empleado {empleado_id} no encontrado en el conjunto de prueba.')
        return
    res = res.iloc[0]

    emp = df_ml[df_ml['empleado_id'] == empleado_id].iloc[0]

    iconos_ret  = {'ALERTA_INMEDIATA':'🚨','REVISAR_COMPENSACION':'💰',
                   'PLAN_BIENESTAR':'🧘','MONITOREAR':'👁️','SIN_ACCION':'✅'}
    iconos_prom = {'CANDIDATO_PROMOCION':'🌟','EN_DESARROLLO':'📈',
                   'NO_ELEGIBLE_AUN':'⏳','REVISION_DISCIPLINARIA':'⛔','NIVEL_MAXIMO':'🏆'}
    sep = '═' * 58

    print(f'\n{sep}')
    print(f'  FICHA DE TRAZABILIDAD RH — {empleado_id}')
    print(f'  Generada: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}')
    print(f'{sep}')

    print(f'\n  PERFIL')
    print(f'  Departamento  : {emp["departamento"]}')
    print(f'  Nivel actual  : {emp["nivel_puesto"]}')
    print(f'  Antigüedad    : {emp["antiguedad_anios"]:.1f} años')

    print(f'\n  MÉTRICAS CLAVE')
    metricas = [
        ('Evaluación de desempeño', emp['evaluacion_desempeno'], '/5',    4, 'mayor'),
        ('Satisfacción laboral',    emp['satisfaccion_laboral'], '/10',   6, 'mayor'),
        ('Horas extra / mes',       emp['horas_extra_mes'],      'h/mes', 25,'menor'),
        ('Capacitaciones en el año',emp['capacitaciones_anio'],  '',      2, 'mayor'),
        ('Incidencias discipl.',    emp['incidencias_disc'],     '',      0, 'igual'),
        ('Meses sin aumento',       emp['meses_ultimo_aumento'], 'meses', 18,'menor'),
    ]
    for nombre, val, unit, umbral, direccion in metricas:
        if direccion == 'mayor': ok = val >= umbral
        elif direccion == 'menor': ok = val <= umbral
        else: ok = val == umbral
        estado = '✅' if ok else '⚠️ '
        print(f'  {estado}  {nombre:<28} {val}{unit}')

    print(f'\n  [1] CAPA NEURAL')
    print(f'  Probabilidad de rotación : {res["prob_rotacion"]:.4f}')
    print(f'  Nivel de riesgo          : {res["riesgo_rotacion"].upper()}')
    print(f'  Señales de alerta        : {res["n_alertas"]} activa(s)')

    print(f'\n  [2] CAPA SIMBÓLICA — DECISIONES')
    print(f'  {iconos_ret.get(res["accion_retencion"],"📋")} Retención  : {res["accion_retencion"]}')
    print(f'     {res["justif_retencion"]}')
    print()
    print(f'  {iconos_prom.get(res["accion_promocion"],"📋")} Promoción  : {res["accion_promocion"]}')
    print(f'     {res["justif_promocion"]}')
    print(f'\n{sep}\n')


# Generar fichas: un candidato a promoción y uno con alerta de retención
candidatos = df_rh[df_rh['accion_promocion'] == 'CANDIDATO_PROMOCION']['empleado_id']
alertas    = df_rh[df_rh['accion_retencion'] == 'ALERTA_INMEDIATA']['empleado_id']

if len(candidatos) > 0:
    ficha_trazabilidad_rh(candidatos.iloc[0], df_rh, df_ml)
if len(alertas) > 0:
    ficha_trazabilidad_rh(alertas.iloc[0], df_rh, df_ml)

## 🏆 Celda 10 — ¿Qué gana RH con la IA Neurosimbólica?

Comparación final entre el enfoque tradicional y el neurosimbólico en el contexto de RH.

In [ ]:
# Casos donde la capa simbólica modifica lo que el modelo solo diría
df_rh['pred_neural_sola'] = df_rh['prob_rotacion'].apply(
    lambda p: 'ALTO RIESGO' if p >= 0.45 else 'BAJO RIESGO'
)

# Empleados con bajo riesgo neural pero que el simbólico marca con acción
falsos_seguros = df_rh[
    (df_rh['pred_neural_sola'] == 'BAJO RIESGO') &
    (df_rh['accion_retencion'].isin(['REVISAR_COMPENSACION', 'ALERTA_INMEDIATA']))
]

# Candidatos a promoción con riesgo alto que serían ignorados sin la capa simbólica
promos_con_riesgo = df_rh[
    (df_rh['accion_promocion'] == 'CANDIDATO_PROMOCION') &
    (df_rh['riesgo_rotacion'].isin(['alto', 'critico']))
]

print('=== VALOR AÑADIDO DE LA IA NEUROSIMBÓLICA EN RH ===')
print(f'\nEmpleados con riesgo bajo según el modelo, pero con alerta simbólica:')
print(f'  → {len(falsos_seguros)} empleados que el modelo solo hubiera ignorado')
print(f'  → La capa simbólica detectó señales de compensación o clima en ellos')

print(f'\nCandidatos a promoción que también tienen riesgo de rotación:')
print(f'  → {len(promos_con_riesgo)} personas: talento valioso en riesgo de salir')
print(f'  → El sistema permite actuar en ambas dimensiones simultáneamente')

print()
print("""
  ╔══════════════════════════════════════════════════════════════╗
  ║  RESUMEN: ¿Qué aporta la IA Neurosimbólica vs. solo ML?     ║
  ╠══════════════════════════════════════════════════════════════╣
  ║  ✅ Decisiones explicables para auditorías y sindicato      ║
  ║  ✅ Detecta patrones sutiles que las reglas solas no verían  ║
  ║  ✅ Política de empresa integrada directamente en el flujo   ║
  ║  ✅ Separa la retención de la promoción (problemas distintos)║
  ║  ✅ El equipo de RH puede actualizar las reglas sin código   ║
  ╚══════════════════════════════════════════════════════════════╝
""")